In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, mean_absolute_error, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
train_df = pd.read_csv('./csvs/train.csv')
test_df = pd.read_csv('./csvs/test.csv')

In [ ]:
train_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [ ]:
train_df.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [ ]:
train_df.describe(include=['O'])

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,VIP,Name
count,8693,8492,8476,8494,8511,8490,8493
unique,8693,3,2,6560,3,2,8473
top,9280_02,Earth,False,G/734/S,TRAPPIST-1e,False,Ankalik Nateansive
freq,1,4602,5439,8,5915,8291,2


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


## Data Cleaning

In [ ]:
# train_df.fillna({"Age": train_df['Age'].mean(), "RoomService": train_df['RoomService'].mean(),"FoodCourt": train_df['FoodCourt'].mean(), "ShoppingMall": train_df['ShoppingMall'].mean(), "Spa": train_df['Spa'].mean(), 'VRDeck': train_df['VRDeck'].mean()}, inplace=True)
# The explicit fillna for numerical columns is being moved into the ColumnTransformer for better pipeline management.

In [ ]:
# test_df.fillna({"Age": test_df['Age'].mean(), "RoomService": test_df['RoomService'].mean(),"FoodCourt": test_df['FoodCourt'].mean(), "ShoppingMall": test_df['ShoppingMall'].mean(), "Spa": test_df['Spa'].mean(), 'VRDeck': test_df['VRDeck'].mean()}, inplace=True)
# The explicit fillna for numerical columns is being moved into the ColumnTransformer for better pipeline management.

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8693 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8693 non-null   float64
 8   FoodCourt     8693 non-null   float64
 9   ShoppingMall  8693 non-null   float64
 10  Spa           8693 non-null   float64
 11  VRDeck        8693 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   4277 non-null   object 
 1   HomePlanet    4190 non-null   object 
 2   CryoSleep     4184 non-null   object 
 3   Cabin         4177 non-null   object 
 4   Destination   4185 non-null   object 
 5   Age           4277 non-null   float64
 6   VIP           4184 non-null   object 
 7   RoomService   4277 non-null   float64
 8   FoodCourt     4277 non-null   float64
 9   ShoppingMall  4277 non-null   float64
 10  Spa           4277 non-null   float64
 11  VRDeck        4277 non-null   float64
 12  Name          4183 non-null   object 
dtypes: float64(6), object(7)
memory usage: 434.5+ KB


In [ ]:
train_df['HomePlanet'].value_counts()

,count
HomePlanet,
Earth,4602
Europa,2131
Mars,1759


In [ ]:
train_df.groupby('HomePlanet')['Transported'].agg(['count', 'mean'])

,count,mean
HomePlanet,,
Earth,4602,0.423946
Europa,2131,0.658846
Mars,1759,0.523024


- passengers from Europa are more likely to get transported to the alternate dimension

In [ ]:
train_df.groupby('CryoSleep')['Transported'].agg(['count', 'mean'])

,count,mean
CryoSleep,,
False,5439,0.328921
True,3037,0.817583


- Passengers confined in their cabins are more likey to be transported

In [ ]:
train_df.groupby('Cabin')['Transported'].agg(['count', 'mean'])

,count,mean
Cabin,,
A/0/P,2,0.5
A/0/S,2,0.0
A/1/S,3,1.0
A/10/P,1,0.0
A/10/S,1,1.0
...,...,...
T/0/P,1,0.0
T/1/P,1,0.0
T/2/P,1,0.0


In [ ]:
train_df.groupby('Destination')['Transported'].agg(['count', 'mean'])

,count,mean
Destination,,
55 Cancri e,1800,0.610000
PSO J318.5-22,796,0.503769
TRAPPIST-1e,5915,0.471175


- passengers whose destination is 55 Cancrie are more likely to be transported to the alternate universe

In [ ]:
train_df['Age'].describe()

,Age
count,8693.000000
mean,28.827930
std,14.339054
min,0.000000
25%,20.000000
50%,27.000000
75%,37.000000
max,79.000000


In [ ]:
(test_df['Age'].describe())

,Age
count,4277.000000
mean,28.658146
std,14.027384
min,0.000000
25%,20.000000
50%,27.000000
75%,37.000000
max,79.000000


In [ ]:
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
age_cut = pd.cut(train_df['Age'], bins=age_bins, labels=['infant', 'pre- teen&teen', 'young_adult', 'mid-age', '40s', '50s', '60s', '70s'], include_lowest=True)
age_cut_test = pd.cut(test_df['Age'], bins=age_bins, labels=['infant', 'pre-teen&teen', 'young adult', 'mid-age', '40s', '50s', '60s', '70s'], include_lowest=True)
train_df.groupby(age_cut)['Transported'].agg(['count', 'mean'])

/tmp/ipykernel_4767/3456957912.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(age_cut)['Transported'].agg(['count', 'mean'])


,count,mean
Age,,
infant,718,0.707521
pre- teen&teen,1717,0.517764
young_adult,2847,0.471373
mid-age,1680,0.468452
40s,994,0.497988
50s,517,0.489362
60s,183,0.491803
70s,37,0.378378


- infants are more likely to be transported to the alternate universe

In [ ]:
train_df['AgeGroup'] = age_cut
test_df['AgeGroup'] = age_cut_test

In [ ]:
train_df['RoomService'].describe()

,RoomService
count,8693.000000
mean,224.687617
std,659.739364
min,0.000000
25%,0.000000
50%,0.000000
75%,78.000000
max,14327.000000


In [ ]:
test_df['RoomService'].describe()

,RoomService
count,4277.000000
mean,219.266269
std,601.162847
min,0.000000
25%,0.000000
50%,0.000000
75%,79.000000
max,11567.000000


In [ ]:
rms_bins = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 14000, 15000]
rms_cut = pd.cut(train_df['RoomService'], bins=rms_bins, labels=['100s','1000s', '2000s', '3000s', '4000s', '5000s', '6000s', '7000s', '8000s', '9000s', '10000s', '14000s'], include_lowest=True)
rms_cut_test = pd.cut(test_df['RoomService'], bins=rms_bins, labels=['100s','1000s', '2000s', '3000s', '4000s', '5000s', '6000s', '7000s', '8000s', '9000s', '10000s', '14000s'], include_lowest=True)
train_df.groupby(rms_cut)['Transported'].agg(['count', 'mean'])

/tmp/ipykernel_4767/3351150918.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(rms_cut)['Transported'].agg(['count', 'mean'])


,count,mean
RoomService,,
100s,8093,0.531323
1000s,394,0.137056
2000s,113,0.168142
3000s,52,0.096154
4000s,13,0.000000
5000s,11,0.000000
6000s,6,0.000000
7000s,2,0.000000
8000s,7,0.000000


- passengers who paid the least amount of room service are more likely to be transported

In [ ]:
train_df['RoomServiceGroup'] = rms_cut
test_df['RoomServiceGroup'] = rms_cut_test
train_df['RoomServiceGroup']

,RoomServiceGroup
0,100s
1,100s
2,100s
3,100s
4,100s
...,...
8688,100s
8689,100s
8690,100s
8691,100s


In [ ]:
train_df['FoodCourt'].describe()

,FoodCourt
count,8693.000000
mean,458.077203
std,1594.434978
min,0.000000
25%,0.000000
50%,0.000000
75%,118.000000
max,29813.000000


In [ ]:
train_df['FoodCourt'].describe()

,FoodCourt
count,8693.000000
mean,458.077203
std,1594.434978
min,0.000000
25%,0.000000
50%,0.000000
75%,118.000000
max,29813.000000


In [ ]:
# f_bins = [0, 2000, 4000, 6000, 8000, 10000, 12000, 14000, 16000, 18000, 20000, 22000, 24000, 26000, 28000, 30000]
f_bins = list(range(0, 32000, 2000))
labels = [f"{i}-{i+2000}" for i in range(0, 30000, 2000)]
foodcourt_cuts = pd.cut(train_df['FoodCourt'], bins=f_bins, labels=labels, include_lowest=True)
foodcourt_cuts_test = pd.cut(test_df['FoodCourt'], bins=f_bins, labels=labels, include_lowest=True)
train_df.groupby(foodcourt_cuts)['Transported'].agg(['count', 'mean'])

/tmp/ipykernel_4767/781527229.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(foodcourt_cuts)['Transported'].agg(['count', 'mean'])


,count,mean
FoodCourt,,
0-2000,8147,0.496502
2000-4000,277,0.530686
4000-6000,127,0.708661
6000-8000,56,0.660714
8000-10000,34,0.588235
10000-12000,20,0.700000
12000-14000,12,0.583333
14000-16000,4,0.750000
16000-18000,10,0.900000


In [ ]:
train_df['FoodCourtGroup'] = foodcourt_cuts
test_df['FoodCourtGroup'] = foodcourt_cuts_test

In [ ]:
train_df["ShoppingMall"].describe()

,ShoppingMall
count,8693.000000
mean,173.729169
std,597.417440
min,0.000000
25%,0.000000
50%,0.000000
75%,45.000000
max,23492.000000


In [ ]:
test_df['ShoppingMall'].describe()

,ShoppingMall
count,4277.000000
mean,177.295525
std,554.357251
min,0.000000
25%,0.000000
50%,0.000000
75%,51.000000
max,8292.000000


In [ ]:
shopping_mall_bins = list(range(0, 26000, 2000))
shopping_mall_bins_test = list(range(0, 10000, 1000))
display(shopping_mall_bins_test)
labels_test = [f'{i}-{i + 1000}' for i in range(0, 9000, 1000)]
labels = [f'{i}-{i + 2000}' for i in range(0, 24000, 2000)]
shopping_mall_cuts = pd.cut(train_df['ShoppingMall'], bins=shopping_mall_bins, labels=labels, include_lowest=True)
shopping_mall_cuts_test = pd.cut(test_df['ShoppingMall'], bins=shopping_mall_bins_test, labels=labels_test, include_lowest=True)
train_df.groupby(shopping_mall_cuts)['Transported'].agg(['count', 'mean'])

[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]

/tmp/ipykernel_4767/2426530560.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(shopping_mall_cuts)['Transported'].agg(['count', 'mean'])


,count,mean
ShoppingMall,,
0-2000,8569,0.500292
2000-4000,94,0.723404
4000-6000,17,0.705882
6000-8000,8,1.000000
8000-10000,1,1.000000
10000-12000,2,0.000000
12000-14000,1,1.000000
14000-16000,0,NaN
16000-18000,0,NaN


- for 6000-10000 and 22000 to 24000, passengers are more likely to be transported

In [ ]:
train_df['ShoppingMallGroup'] = shopping_mall_cuts
test_df['ShoppingMallGroup'] = shopping_mall_cuts_test

In [ ]:
train_df['Spa'].describe()

,Spa
count,8693.000000
mean,311.138778
std,1124.675871
min,0.000000
25%,0.000000
50%,0.000000
75%,89.000000
max,22408.000000


In [ ]:
test_df['Spa'].describe()

,Spa
count,4277.000000
mean,303.052443
std,1103.913087
min,0.000000
25%,0.000000
50%,0.000000
75%,83.000000
max,19844.000000


In [ ]:
spa_bins = list(range(0, 26000, 2000))
labels = [f'{i}-{i + 2000}' for i in range(0, 24000, 2000)]
spa_cuts = pd.cut(train_df['Spa'], bins=spa_bins, labels=labels, include_lowest=True)
spa_cuts_test = pd.cut(test_df['Spa'], bins=spa_bins, labels=labels, include_lowest=True)
train_df.groupby(spa_cuts)['Transported'].agg(['count', 'mean'])

/tmp/ipykernel_4767/3857064747.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(spa_cuts)['Transported'].agg(['count', 'mean'])


,count,mean
Spa,,
0-2000,8354,0.521547
2000-4000,183,0.109290
4000-6000,87,0.011494
6000-8000,28,0.000000
8000-10000,17,0.000000
10000-12000,8,0.000000
12000-14000,8,0.000000
14000-16000,4,0.000000
16000-18000,2,0.000000


In [ ]:
train_df['SpaGroup'] = spa_cuts
test_df['SpaGroup'] = spa_cuts_test

In [ ]:
train_df['VRDeck'].describe()

,VRDeck
count,8693.000000
mean,304.854791
std,1133.259049
min,0.000000
25%,0.000000
50%,0.000000
75%,71.000000
max,24133.000000


In [ ]:
test_df['VRDeck'].describe()

,VRDeck
count,4277.000000
mean,310.710031
std,1235.274606
min,0.000000
25%,0.000000
50%,0.000000
75%,53.000000
max,22272.000000


In [ ]:
vr_bins = list(range(0, 28000, 2000))
# display(vr_bins)
labels = [f'{i}-{i + 2000}' for i in range(0, 25000, 2000)]
vr_cuts = pd.cut(train_df['VRDeck'], bins=vr_bins, labels=labels, include_lowest=True)
vr_cuts_test = pd.cut(test_df['VRDeck'], bins=vr_bins, labels=labels, include_lowest=True)
train_df.groupby(vr_cuts)['Transported'].agg(['count', 'mean'])

/tmp/ipykernel_4767/63927243.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_df.groupby(vr_cuts)['Transported'].agg(['count', 'mean'])


,count,mean
VRDeck,,
0-2000,8371,0.520368
2000-4000,173,0.115607
4000-6000,70,0.028571
6000-8000,35,0.000000
8000-10000,16,0.000000
10000-12000,14,0.000000
12000-14000,8,0.000000
14000-16000,1,0.000000
16000-18000,3,0.000000


In [ ]:
train_df['VRDeckGroup'] = vr_cuts
test_df['VRDeckGroup'] = vr_cuts_test

In [ ]:
test_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,AgeGroup,RoomServiceGroup,FoodCourtGroup,ShoppingMallGroup,SpaGroup,VRDeckGroup
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.000000,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning,young adult,100s,0-2000,0-1000,0-2000,0-2000
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.000000,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers,pre-teen&teen,100s,0-2000,0-1000,2000-4000,0-2000
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.000000,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus,mid-age,100s,0-2000,0-1000,0-2000,0-2000
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.000000,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter,mid-age,100s,6000-8000,0-1000,0-2000,0-2000
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.000000,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez,pre-teen&teen,100s,0-2000,0-1000,0-2000,0-2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.000000,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter,mid-age,100s,0-2000,0-1000,0-2000,0-2000
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.000000,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron,40s,100s,0-2000,0-1000,0-2000,0-2000
4274,9271_01,Mars,True,D/296/P,55 Cancri e,28.658146,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore,young adult,100s,0-2000,0-1000,0-2000,0-2000
4275,9273_01,Europa,False,D/297/P,NaN,28.658146,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale,young adult,100s,2000-4000,0-1000,0-2000,0-2000


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   PassengerId        8693 non-null   object  
 1   HomePlanet         8492 non-null   object  
 2   CryoSleep          8476 non-null   object  
 3   Cabin              8494 non-null   object  
 4   Destination        8511 non-null   object  
 5   Age                8693 non-null   float64 
 6   VIP                8490 non-null   object  
 7   RoomService        8693 non-null   float64 
 8   FoodCourt          8693 non-null   float64 
 9   ShoppingMall       8693 non-null   float64 
 10  Spa                8693 non-null   float64 
 11  VRDeck             8693 non-null   float64 
 12  Name               8493 non-null   object  
 13  Transported        8693 non-null   bool    
 14  AgeGroup           8693 non-null   category
 15  RoomServiceGroup   8693 non-null   category
 16  FoodCo

In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   PassengerId        4277 non-null   object  
 1   HomePlanet         4190 non-null   object  
 2   CryoSleep          4184 non-null   object  
 3   Cabin              4177 non-null   object  
 4   Destination        4185 non-null   object  
 5   Age                4277 non-null   float64 
 6   VIP                4184 non-null   object  
 7   RoomService        4277 non-null   float64 
 8   FoodCourt          4277 non-null   float64 
 9   ShoppingMall       4277 non-null   float64 
 10  Spa                4277 non-null   float64 
 11  VRDeck             4277 non-null   float64 
 12  Name               4183 non-null   object  
 13  AgeGroup           4277 non-null   category
 14  RoomServiceGroup   4277 non-null   category
 15  FoodCourtGroup     4277 non-null   category
 16  Shoppi

In [ ]:
# X = train_df['Transported']
# y = train_df.drop(columns=['Transported'])
train_df['CryoSleep'] = (train_df['CryoSleep'] == True).astype(int)
train_df['Transported'] = (train_df['Transported'] == True).astype(int)
train_df['VIP'] = (train_df['VIP'] == True).astype(int)

test_df['CryoSleep'] = (test_df['CryoSleep'] == True).astype(int)
test_df['VIP'] = (test_df['VIP'] == True).astype(int)

In [ ]:
test_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,AgeGroup,RoomServiceGroup,FoodCourtGroup,ShoppingMallGroup,SpaGroup,VRDeckGroup
0,0013_01,Earth,1,G/3/S,TRAPPIST-1e,27.000000,0,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning,young adult,100s,0-2000,0-1000,0-2000,0-2000
1,0018_01,Earth,0,F/4/S,TRAPPIST-1e,19.000000,0,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers,pre-teen&teen,100s,0-2000,0-1000,2000-4000,0-2000
2,0019_01,Europa,1,C/0/S,55 Cancri e,31.000000,0,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus,mid-age,100s,0-2000,0-1000,0-2000,0-2000
3,0021_01,Europa,0,C/1/S,TRAPPIST-1e,38.000000,0,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter,mid-age,100s,6000-8000,0-1000,0-2000,0-2000
4,0023_01,Earth,0,F/5/S,TRAPPIST-1e,20.000000,0,10.0,0.0,635.0,0.0,0.0,Brence Harperez,pre-teen&teen,100s,0-2000,0-1000,0-2000,0-2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,1,G/1496/S,TRAPPIST-1e,34.000000,0,0.0,0.0,0.0,0.0,0.0,Jeron Peter,mid-age,100s,0-2000,0-1000,0-2000,0-2000
4273,9269_01,Earth,0,NaN,TRAPPIST-1e,42.000000,0,0.0,847.0,17.0,10.0,144.0,Matty Scheron,40s,100s,0-2000,0-1000,0-2000,0-2000
4274,9271_01,Mars,1,D/296/P,55 Cancri e,28.658146,0,0.0,0.0,0.0,0.0,0.0,Jayrin Pore,young adult,100s,0-2000,0-1000,0-2000,0-2000
4275,9273_01,Europa,0,D/297/P,NaN,28.658146,0,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale,young adult,100s,2000-4000,0-1000,0-2000,0-2000


- Cabin (deck/num/side)

In [ ]:
train_df['Cabin'].str.split('/', expand=True)[1]

,1
0,0
1,0
2,0
3,0
4,1
...,...
8688,98
8689,1499
8690,1500
8691,608


In [ ]:
cabin_parts = train_df['Cabin'].str.split('/', expand=True)
train_df['Deck'] = cabin_parts[0]
test_df['Deck'] = test_df['Cabin'].str.split('/', expand=True)[0]
train_df['CabinNum'] = cabin_parts[1]
test_df['CabinNum'] = test_df['Cabin'].str.split('/', expand=True)[1]
train_df['Side'] = cabin_parts[2]
test_df['Side'] = test_df['Cabin'].str.split('/', expand=True)[2]

In [ ]:
train_df['Deck'].value_counts()

,count
Deck,
F,2794
G,2559
E,876
B,779
C,747
D,478
A,256
T,5


In [ ]:
train_df.groupby('Deck', as_index=False)['Transported'].agg(['count', 'mean'])

,Deck,count,mean
0,A,256,0.496094
1,B,779,0.734275
2,C,747,0.680054
3,D,478,0.433054
4,E,876,0.357306
5,F,2794,0.439871
6,G,2559,0.516217
7,T,5,0.200000


In [ ]:
train_df['CabinNum'].value_counts()

,count
CabinNum,
82,28
86,22
19,22
176,21
56,21
...,...
1839,1
1848,1
1847,1


In [ ]:
train_df['CabinNumCounts'] = train_df.groupby(['CabinNum'])['CabinNum'].transform('count')
test_df['CabinNumCounts'] = test_df.groupby(['CabinNum'])['CabinNum'].transform('count')

In [ ]:
train_df.head(20)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,AgeGroup,RoomServiceGroup,FoodCourtGroup,ShoppingMallGroup,SpaGroup,VRDeckGroup,Deck,CabinNum,Side,CabinNumCounts
0,0001_01,Europa,0,B/0/P,TRAPPIST-1e,39.0,0,0.0,0.0,0.000000,0.0,0.000000,Maham Ofracculy,0,mid-age,100s,0-2000,0-2000,0-2000,0-2000,B,0,P,18.0
1,0002_01,Earth,0,F/0/S,TRAPPIST-1e,24.0,0,109.0,9.0,25.000000,549.0,44.000000,Juanna Vines,1,young_adult,100s,0-2000,0-2000,0-2000,0-2000,F,0,S,18.0
2,0003_01,Europa,0,A/0/S,TRAPPIST-1e,58.0,1,43.0,3576.0,0.000000,6715.0,49.000000,Altark Susent,0,50s,100s,2000-4000,0-2000,6000-8000,0-2000,A,0,S,18.0
3,0003_02,Europa,0,A/0/S,TRAPPIST-1e,33.0,0,0.0,1283.0,371.000000,3329.0,193.000000,Solam Susent,0,mid-age,100s,0-2000,0-2000,2000-4000,0-2000,A,0,S,18.0
4,0004_01,Earth,0,F/1/S,TRAPPIST-1e,16.0,0,303.0,70.0,151.000000,565.0,2.000000,Willy Santantines,1,pre- teen&teen,100s,0-2000,0-2000,0-2000,0-2000,F,1,S,15.0
5,0005_01,Earth,0,F/0/P,PSO J318.5-22,44.0,0,0.0,483.0,0.000000,291.0,0.000000,Sandie Hinetthews,1,40s,100s,0-2000,0-2000,0-2000,0-2000,F,0,P,18.0
6,0006_01,Earth,0,F/2/S,TRAPPIST-1e,26.0,0,42.0,1539.0,3.000000,0.0,0.000000,Billex Jacostaffey,1,young_adult,100s,0-2000,0-2000,0-2000,0-2000,F,2,S,11.0
7,0006_02,Earth,1,G/0/S,TRAPPIST-1e,28.0,0,0.0,0.0,0.000000,0.0,304.854791,Candra Jacostaffey,1,young_adult,100s,0-2000,0-2000,0-2000,0-2000,G,0,S,18.0
8,0007_01,Earth,0,F/3/S,TRAPPIST-1e,35.0,0,0.0,785.0,17.000000,216.0,0.000000,Andona Beston,1,mid-age,100s,0-2000,0-2000,0-2000,0-2000,F,3,S,16.0
9,0008_01,Europa,1,B/1/P,55 Cancri e,14.0,0,0.0,0.0,0.000000,0.0,0.000000,Erraiam Flatic,1,pre- teen&teen,100s,0-2000,0-2000,0-2000,0-2000,B,1,P,15.0


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   PassengerId        8693 non-null   object  
 1   HomePlanet         8492 non-null   object  
 2   CryoSleep          8693 non-null   int64   
 3   Cabin              8494 non-null   object  
 4   Destination        8511 non-null   object  
 5   Age                8693 non-null   float64 
 6   VIP                8693 non-null   int64   
 7   RoomService        8693 non-null   float64 
 8   FoodCourt          8693 non-null   float64 
 9   ShoppingMall       8693 non-null   float64 
 10  Spa                8693 non-null   float64 
 11  VRDeck             8693 non-null   float64 
 12  Name               8493 non-null   object  
 13  Transported        8693 non-null   int64   
 14  AgeGroup           8693 non-null   category
 15  RoomServiceGroup   8693 non-null   category
 16  FoodCo

In [ ]:
X = train_df.drop(columns=['Transported'])
y = train_df['Transported']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Manual filling of categorical columns is removed as it will be handled in the ColumnTransformer.
# Only numerical NaNs from CabinNumCounts are filled here, though this can also be moved into the pipeline if desired.
X_train['CabinNumCounts'] = X_train['CabinNumCounts'].fillna(X_train['CabinNumCounts'].median())
X_test['CabinNumCounts'] = X_test['CabinNumCounts'].fillna(X_test['CabinNumCounts'].median())

In [ ]:
X_train

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,AgeGroup,RoomServiceGroup,FoodCourtGroup,ShoppingMallGroup,SpaGroup,VRDeckGroup,Deck,CabinNum,Side,CabinNumCounts
5623,5981_01,Mars,0,F/1140/S,TRAPPIST-1e,27.0,0,441.0,0.0,397.0,471.0,0.0,Harz Quart,young_adult,100s,0-2000,0-2000,0-2000,0-2000,F,1140,S,5.0
5253,5606_01,Europa,1,B/213/S,55 Cancri e,45.0,0,0.0,0.0,0.0,0.0,0.0,Algor Paterpad,40s,100s,0-2000,0-2000,0-2000,0-2000,B,213,S,12.0
478,0515_01,Europa,1,B/20/S,TRAPPIST-1e,50.0,0,0.0,0.0,0.0,0.0,0.0,Alramix Swinvul,40s,100s,0-2000,0-2000,0-2000,0-2000,B,20,S,15.0
1352,1425_02,Earth,1,G/220/P,TRAPPIST-1e,1.0,0,0.0,0.0,0.0,0.0,0.0,Mael Adavisons,infant,100s,0-2000,0-2000,0-2000,0-2000,G,220,P,11.0
5344,5713_01,Earth,0,G/915/P,TRAPPIST-1e,42.0,0,0.0,29.0,317.0,434.0,45.0,Lawren Blangibson,40s,100s,0-2000,0-2000,0-2000,0-2000,G,915,P,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,6076_01,Earth,0,G/988/S,TRAPPIST-1e,18.0,0,14.0,2.0,144.0,610.0,0.0,Therry Cames,pre- teen&teen,100s,0-2000,0-2000,0-2000,0-2000,G,988,S,4.0
5191,5537_01,Mars,0,F/1063/S,TRAPPIST-1e,50.0,0,690.0,0.0,30.0,762.0,428.0,Herms Bancy,40s,100s,0-2000,0-2000,0-2000,0-2000,F,1063,S,4.0
5390,5756_06,Earth,0,F/1194/P,PSO J318.5-22,22.0,0,158.0,0.0,476.0,0.0,26.0,Karena Briggston,young_adult,100s,0-2000,0-2000,0-2000,0-2000,F,1194,P,10.0
860,0925_01,Mars,0,F/191/P,TRAPPIST-1e,34.0,0,379.0,0.0,1626.0,0.0,0.0,Skix Kraie,mid-age,100s,0-2000,0-2000,0-2000,0-2000,F,191,P,10.0


# Pipeline

In [ ]:
ohe_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
ode_cols=['AgeGroup', 'RoomServiceGroup', 'FoodCourtGroup', 'ShoppingMallGroup', 'SpaGroup', 'VRDeckGroup']
SI = SimpleImputer()

In [ ]:
# The ohe_pipeline and ode_pipeline are now integrated directly into the ColumnTransformer for more explicit control over imputation order.
# ohe_pipeline = Pipeline([
#     ('ohe', OneHotEncoder(handle_unknown='ignore')),
#     ('impute', SimpleImputer(strategy='most_frequent'))
# ])

# ode_pipeline = Pipeline([
#     ('ode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, dtype=float)),
#     ('impute', SimpleImputer(strategy='median'))
# ])

In [454]:
numeric_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNumCounts']
categorical_features_ohe = ['HomePlanet', 'Destination', 'Deck', 'Side']
ordinal_features = ['AgeGroup', 'RoomServiceGroup', 'FoodCourtGroup', 'ShoppingMallGroup', 'SpaGroup', 'VRDeckGroup']
binary_features = ['CryoSleep', 'VIP']

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Impute numerical NaNs with median
    ('scaler', RobustScaler()) # Scale numerical features
])

categorical_transformer_ohe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), # Impute categorical NaNs with 'Unknown' constant
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute ordinal NaNs with most frequent
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)) # Encode ordinal features, -1 for unknown
])

# Create the ColumnTransformer to apply different transformations to different columns
column_pipeline = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat_ohe', categorical_transformer_ohe, categorical_features_ohe),
        ('ord', ordinal_transformer, ordinal_features),
        ('passthrough', 'passthrough', binary_features) # Binary features remain as is
    ],
    remainder='drop', # Drop any other columns (like PassengerId, Name, Cabin, CabinNum) not explicitly processed
    n_jobs=-1
)

## Models

### KNN

In [443]:
knn_pipeline = Pipeline([
    ('sampler', RandomOverSampler(random_state=42)),
    ('knn', KNeighborsClassifier())
])

In [444]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 20],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

In [445]:
cv = StratifiedKFold(n_splits=5)

In [446]:
knn_grid = GridSearchCV(knn_pipeline, param_grid, cv=cv, scoring='recall')

In [455]:
final_knn_pipeline = make_pipeline(column_pipeline, knn_grid)

In [456]:
final_knn_pipeline = make_pipeline(column_pipeline, knn_grid)
final_knn_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')...
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=Pipeline(steps=[('sampler',
                                                         RandomOverSampler(random_state=42)),
                                                        ('knn',
                                                         KNeighborsClassifier())]),
                              param_grid={'knn__n_neighbors': [3, 5, 7, 9, 11,
                                                               15, 20],
                                          'knn__p': [1, 2],
                                          'knn__weights': ['uniform',
                                                           'distance']},
                              scoring='recall'))])

In [457]:
y_pred = final_knn_pipeline.predict(X_test)

In [458]:
print("Best Estimator: ", knn_grid.best_estimator_)
print('Best Score: ', knn_grid.best_score_)

Best Estimator:  Pipeline(steps=[('sampler', RandomOverSampler(random_state=42)),
                ('knn', KNeighborsClassifier(n_neighbors=15, p=1))])
Best Score:  0.8070616646542031


In [459]:
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.7856485740570377
              precision    recall  f1-score   support

           0       0.80      0.77      0.78      1082
           1       0.78      0.81      0.79      1092

    accuracy                           0.79      2174
   macro avg       0.79      0.79      0.79      2174
weighted avg       0.79      0.79      0.79      2174



In [460]:
mean_absolute_error(y_test, y_pred)

0.21435142594296228

In [461]:
roc_auc_score(y_test, y_pred)

np.float64(0.7855551718767986)

In [428]:
y_test.value_counts(normalize=True)

,proportion
Transported,
1,0.5023
0,0.4977


## Naive Bayes

In [462]:
gnb_pipeline = Pipeline([
    ('sampler', RandomOverSampler(random_state=42)),
    ('gnb', GaussianNB())
])

In [463]:
param_grid = {
    'gnb__var_smoothing': [0.0000001, 0.000000001, 0.0000000001]
}

In [464]:
gnb_grid = GridSearchCV(gnb_pipeline, param_grid, cv=cv, scoring='f1')

In [465]:
final_gnb_pipeline = make_pipeline(column_pipeline, gnb_grid)

In [466]:
final_gnb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')...
                                                   'ShoppingMallGroup',
                                                   'SpaGroup', 'VRDeckGroup']),
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=Pipeline(steps=[('sampler',
                                                         RandomOverSampler(random_state=42)),
                                                        ('gnb', GaussianNB())]),
                              param_grid={'gnb__var_smoothing': [1e-07, 1e-09,
                                                                 1e-10]},
                              scoring='f1'))])

In [467]:
y_pred = final_gnb_pipeline.predict(X_test)

In [468]:
print("Best Estimator: ", gnb_grid.best_estimator_)
print("Best Score: ", gnb_grid.best_score_)

Best Estimator:  Pipeline(steps=[('sampler', RandomOverSampler(random_state=42)),
                ('gnb', GaussianNB(var_smoothing=1e-07))])
Best Score:  0.7374837480929257


In [469]:
print(accuracy_score(y_test, y_pred))

0.6678932842686293


In [470]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.37      0.53      1082
           1       0.61      0.96      0.74      1092

    accuracy                           0.67      2174
   macro avg       0.75      0.67      0.64      2174
weighted avg       0.75      0.67      0.64      2174



## Logistic Regression

In [474]:
lgr_pipeline = Pipeline([
    ('sample', RandomOverSampler(random_state=42)),
    ('lgr', LogisticRegression( solver='saga',  # required for l1 / elasticnet
    max_iter=1000))
])

In [476]:
param_grid = {
"lgr__C": [0.001, 0.01, 0.1, 1, 10],
"lgr__l1_ratio": [0, 0.5, 1],  # replaces penalty
}

In [477]:
lgr_cv = GridSearchCV(lgr_pipeline, param_grid, cv=cv)

In [478]:
final_lgr_pipeline = make_pipeline(column_pipeline, lgr_cv)

In [479]:
final_lgr_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: Convergen

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')...
                                                   'SpaGroup', 'VRDeckGroup']),
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=Pipeline(steps=[('sample',
                                                         RandomOverSampler(random_state=42)),
                                                        ('lgr',
                                                         LogisticRegression(max_iter=1000,
                                                                            solver='saga'))]),
                              param_grid={'lgr__C': [0.001, 0.01, 0.1, 1, 10],
                                          'lgr__l1_ratio': [0, 0.5, 1]}))])

In [480]:
y_pred_lgr = final_lgr_pipeline.predict(X_test)
print("Best Estimator: ", lgr_cv.best_estimator_)
print("Best score: ", lgr_cv.best_score_)
print(classification_report(y_test, y_pred_lgr))
print(accuracy_score(y_test, y_pred_lgr))

Best Estimator:  Pipeline(steps=[('sample', RandomOverSampler(random_state=42)),
                ('lgr',
                 LogisticRegression(C=0.01, l1_ratio=0, max_iter=1000,
                                    solver='saga'))])
Best score:  0.795829703986553
              precision    recall  f1-score   support

           0       0.82      0.72      0.77      1082
           1       0.76      0.84      0.80      1092

    accuracy                           0.78      2174
   macro avg       0.79      0.78      0.78      2174
weighted avg       0.79      0.78      0.78      2174

0.7833486660533578


In [482]:
y_pred = final_lgr_pipeline.predict(X_test)

In [483]:
print("Best Estimator: ", lgr_cv.best_estimator_)
print("Best score: ", lgr_cv.best_score_)

Best Estimator:  Pipeline(steps=[('sample', RandomOverSampler(random_state=42)),
                ('lgr',
                 LogisticRegression(C=0.01, l1_ratio=0, max_iter=1000,
                                    solver='saga'))])
Best score:  0.795829703986553


In [484]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.72      0.77      1082
           1       0.76      0.84      0.80      1092

    accuracy                           0.78      2174
   macro avg       0.79      0.78      0.78      2174
weighted avg       0.79      0.78      0.78      2174



In [485]:
print(accuracy_score(y_test, y_pred))

0.7833486660533578


## SVM

In [488]:
svc_pipeline = Pipeline([
    ('svc', SVC())
])

In [489]:
param_grid = {
    'svc__C': [0.001, 0.01, 0.1, 1.0, 10, 100, 1000],
}

In [490]:
svc_cv = GridSearchCV(svc_pipeline, param_grid, cv=cv)

In [491]:
final_svc_pipeline = make_pipeline(column_pipeline, svc_cv)

In [492]:
final_svc_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')...
                                                                                  unknown_value=-1))]),
                                                  ['AgeGroup',
                                                   'RoomServiceGroup',
                                                   'FoodCourtGroup',
                                                   'ShoppingMallGroup',
                                                   'SpaGroup', 'VRDeckGroup']),
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=Pipeline(steps=[('svc', SVC())]),
                              param_grid={'svc__C': [0.001, 0.01, 0.1, 1.0, 10,
                                                     100, 1000]}))])

In [494]:
print("Best Estimator:", svc_cv.best_estimator_)
print("Best Score: ", svc_cv.best_score_)

Best Estimator: Pipeline(steps=[('svc', SVC(C=10))])
Best Score:  0.8001251241825141


In [495]:
y_pred = final_svc_pipeline.predict(X_test)

In [496]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.72      0.77      1082
           1       0.76      0.85      0.80      1092

    accuracy                           0.79      2174
   macro avg       0.79      0.79      0.79      2174
weighted avg       0.79      0.79      0.79      2174



In [497]:
print(accuracy_score(y_test, y_pred))

0.7865685372585096


## Tree Models


## RandomForest

In [498]:
rfc_pipeline = RandomForestClassifier()

In [499]:
param_grid = {
    'n_estimators': [100, 150, 200],
    'min_samples_split': [5, 10, 15],
    'max_depth': [8, 9, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy', 'log_loss']
}

In [500]:
CV_rfc = GridSearchCV(estimator=rfc_pipeline, param_grid=param_grid, cv=cv)

In [501]:
final_rfc_pipeline = make_pipeline(column_pipeline, CV_rfc)
final_rfc_pipeline.fit(X_train, y_train) # 33 min

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneH...
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=RandomForestClassifier(),
                              param_grid={'criterion': ['gini', 'entropy',
                                                        'log_loss'],
                                          'max_depth': [8, 9, 10, 15, 20],
                                          'min_samples_leaf': [1, 2, 4],
                                          'min_samples_split': [5, 10, 15],
                                          'n_estimators': [100, 150, 200]}))])

In [502]:
print(f"Best Estimator: {CV_rfc.best_estimator_}")
print(f"Best Parameters: {CV_rfc.best_params_}")
print(f"Best Score: {CV_rfc.best_score_}")

Best Estimator: RandomForestClassifier(max_depth=20, min_samples_leaf=2, min_samples_split=10,
                       n_estimators=150)
Best Parameters: {'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 150}
Best Score: 0.8093261656677135


In [503]:
y_pred = final_rfc_pipeline.predict(X_test)

In [504]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.79      0.79      1082
           1       0.79      0.80      0.80      1092

    accuracy                           0.79      2174
   macro avg       0.79      0.79      0.79      2174
weighted avg       0.79      0.79      0.79      2174



In [505]:
print(accuracy_score(y_test, y_pred))

0.7943882244710212


### Decision Trees

In [506]:
dtc = DecisionTreeClassifier()

In [507]:
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'min_samples_split': [2, 5, 7, 9],
    'min_samples_leaf': [1, 3, 5, 9, 11],
    'max_depth': [10, 20, 70, 100]
}

In [508]:
dtc_cv = GridSearchCV(estimator=dtc, param_grid=param_grid, cv=cv)

In [509]:
final_dtc_cv = make_pipeline(column_pipeline, dtc_cv)
final_dtc_cv.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(n_jobs=-1,
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumCounts']),
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneH...
                                                   'SpaGroup', 'VRDeckGroup']),
                                                 ('passthrough', 'passthrough',
                                                  ['CryoSleep', 'VIP'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=DecisionTreeClassifier(),
                              param_grid={'criterion': ['gini', 'entropy',
                                                        'log_loss'],
                                          'max_depth': [10, 20, 70, 100],
                                          'min_samples_leaf': [1, 3, 5, 9, 11],
                                          'min_samples_split': [2, 5, 7, 9]}))])

In [510]:
y_pred = final_dtc_cv.predict(X_test)

In [511]:
print(f"Best Estimator: {dtc_cv.best_estimator_}")
print(f"Best Parameters: {dtc_cv.best_params_}")
print(f"Best Score: {dtc_cv.best_score_}")

Best Estimator: DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=9,
                       min_samples_split=5)
Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 9, 'min_samples_split': 5}
Best Score: 0.7829436788157579


In [512]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.76      0.77      1082
           1       0.77      0.79      0.78      1092

    accuracy                           0.77      2174
   macro avg       0.77      0.77      0.77      2174
weighted avg       0.77      0.77      0.77      2174



In [513]:
print(accuracy_score(y_test, y_pred))

0.7727690892364305


# Test Data Fit and Submission

In [514]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   PassengerId        4277 non-null   object  
 1   HomePlanet         4190 non-null   object  
 2   CryoSleep          4277 non-null   int64   
 3   Cabin              4177 non-null   object  
 4   Destination        4185 non-null   object  
 5   Age                4277 non-null   float64 
 6   VIP                4277 non-null   int64   
 7   RoomService        4277 non-null   float64 
 8   FoodCourt          4277 non-null   float64 
 9   ShoppingMall       4277 non-null   float64 
 10  Spa                4277 non-null   float64 
 11  VRDeck             4277 non-null   float64 
 12  Name               4183 non-null   object  
 13  AgeGroup           4277 non-null   category
 14  RoomServiceGroup   4277 non-null   category
 15  FoodCourtGroup     4277 non-null   category
 16  Shoppi

In [ ]:
X_test

In [ ]:
# (test_df['RoomService'] == 'NaN').sum()
for col in test_df.columns:
    print(col.upper())
    print((test_df[col] == 'NaN').sum())

In [515]:
y_pred_knn = final_knn_pipeline.predict(test_df)
y_pred_lgr = final_lgr_pipeline.predict(test_df)
y_pred_gnb = final_gnb_pipeline.predict(test_df)
y_pred_svc = final_svc_pipeline.predict(test_df)
y_pred_rfc = final_rfc_pipeline.predict(test_df)
y_pred_dtc = final_dtc_cv.predict(test_df)

In [516]:
y_pred_knn

array([1, 0, 1, ..., 1, 1, 1])

In [517]:
y_pred_knn = (y_pred_knn == 1).astype(str)
y_pred_lgr = (y_pred_lgr == 1).astype(str)
y_pred_gnb = (y_pred_gnb == 1).astype(str)
y_pred_svc = (y_pred_svc == 1).astype(str)
y_pred_rfc = (y_pred_rfc == 1).astype(str)
y_pred_dtc = (y_pred_dtc == 1).astype(str)


In [518]:
y_pred_knn

array(['True', 'False', 'True', ..., 'True', 'True', 'True'], dtype='<U5')

 ### Final CSVs

In [520]:
knn_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_knn})
lgr_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_lgr})
gnb_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_gnb})
svc_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_svc})
rfc_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_rfc})
dtc_csv = pd.DataFrame({'PassengerId': test_df['PassengerId'], 'Transported': y_pred_dtc})

In [521]:
knn_csv

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
...,...,...
4272,9266_02,True
4273,9269_01,False
4274,9271_01,True
4275,9273_01,True


#### CSVs

In [522]:
knn_csv.to_csv('knn_submission.csv', index=False)
lgr_csv.to_csv('lgr_submission.csv', index=False)
gnb_csv.to_csv('gnb_submission.csv', index=False)
svc_csv.to_csv('svc_submission.csv', index=False)
rfc_csv.to_csv('rfc_submission.csv', index=False)
dtc_csv.to_csv('dtc_submission.csv', index=False)